In [8]:
import pandas as pd
import os
from glob import glob

files = glob("../../data/raw/*.xlsx")

metadata = []

for idx, file in enumerate(files, start=1):

    try:
        # Read raw sheet to locate the header row
        raw = pd.read_excel(file, header=None)

        # Find the row containing "Serial"
        header_row = raw[raw.iloc[:, 0].astype(str).str.strip() == "Serial"].index[0]

        # Read again using the detected header row
        df = pd.read_excel(file, header=header_row)

        # Clean column names
        df.columns = df.columns.astype(str).str.strip()

        station_name = (
            os.path.basename(file)
            .replace("-Jan 1, 2022-May 30, 2026.xlsx", "")
            .replace(".xlsx", "")
            .strip()
        )

        metadata.append({
            "station_id": f"WB{idx:03d}",
            "station_name": station_name,
            "district": df["District"].iloc[0],
            "latitude": df["Latitude"].iloc[0],
            "longitude": df["Longitude"].iloc[0],
            "start_date": pd.to_datetime(df["Date"]).min(),
            "end_date": pd.to_datetime(df["Date"]).max(),
            "total_records": len(df),
            "data_source": "WBPCB",
            "city": df["Location"].iloc[0] if "Location" in df.columns else "Kolkata",
            "state": "West Bengal"
        })

    except Exception as e:
        print(f"Error processing {os.path.basename(file)}: {e}")

station_metadata = pd.DataFrame(metadata)

# Ensure output directory exists
os.makedirs("../../data/metadata", exist_ok=True)

station_metadata.to_csv(
    "../../data/metadata/station_metadata.csv",
    index=False
)

print(f"Processed {len(station_metadata)} stations")
station_metadata.head()

Processed 17 stations


,station_id,station_name,district,latitude,longitude,start_date,end_date,total_records,data_source,city,state
0,WB001,Avidipta Housing Complex,Kolkata,22.493217,88.398310,2022-03-14,2026-05-30,36115,WBPCB,Avidipta Housing Complex,West Bengal
1,WB002,"Ballygunge Campus, C.U",Kolkata,22.526701,88.363210,2022-10-15,2026-05-30,27602,WBPCB,"Ballygunge Campus, C.U",West Bengal
2,WB003,Bethune College,Kolkata,22.588510,88.367807,2024-03-01,2026-05-30,15838,WBPCB,Bethune College,West Bengal
3,WB004,Dhapa Lock Pumping Station,Kolkata,22.558027,88.409980,2023-01-11,2026-05-30,27182,WBPCB,Dhapa Lock Pumping Station,West Bengal
4,WB005,"East Calcutta Girls College, Lake Town",Kolkata,22.601583,88.404556,2022-01-01,2026-05-30,36539,WBPCB,"East Calcutta Girls College, Lake Town",West Bengal


In [7]:
for file in files:
   
    df = pd.read_excel(file, header=7)

    print(df.columns.tolist())
    break

['Serial', 'Location', 'District', 'Date', 'Hour', 'Latitude', 'Longitude', 'AQI', 'PM 2.5 AVG (µg/m³)', 'PM 10 AVG (µg/m³)', 'REL HUMI (%)', 'TEMPERATURE (°C)', 'Wind Direction Avg (°)', 'Wind Speed Avg', 'Wind Speed Minimum (in m/s)', 'Wind Speed Maximum (in m/s)']


In [9]:
import pandas as pd
import os
from glob import glob

# =====================================================
# Load all WBPCB station files
# =====================================================

files = glob("../../data/raw/*.xlsx")

metadata = []

for idx, file in enumerate(sorted(files), start=1):

    print(f"[{idx}/{len(files)}] Processing {os.path.basename(file)}")

    # -------------------------------------------------
    # Detect actual header row
    # -------------------------------------------------
    raw = pd.read_excel(file, header=None)

    header_row = raw[
        raw.iloc[:, 0].astype(str).str.strip() == "Serial"
    ].index[0]

    # -------------------------------------------------
    # Read data using detected header
    # -------------------------------------------------
    df = pd.read_excel(file, header=header_row)

    # Clean column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
    )

    station_name = (
        os.path.basename(file)
        .replace("-Jan 1, 2022-May 30, 2026.xlsx", "")
        .replace(".xlsx", "")
        .strip()
    )

    metadata.append({
        "station_id": f"WB{idx:03d}",
        "station_name": station_name,
        "location": df["Location"].iloc[0],
        "district": df["District"].iloc[0],
        "latitude": float(df["Latitude"].iloc[0]),
        "longitude": float(df["Longitude"].iloc[0]),
        "start_date": pd.to_datetime(df["Date"]).min(),
        "end_date": pd.to_datetime(df["Date"]).max(),
        "total_records": len(df),
        "data_source": "WBPCB",
        "state": "West Bengal",
        "schema_version": "v1.0"
    })

# =====================================================
# Create metadata dataframe
# =====================================================

station_metadata = pd.DataFrame(metadata)

# Ensure output directory exists
os.makedirs("../../data/metadata", exist_ok=True)

station_metadata.to_csv(
    "../../data/metadata/station_metadata.csv",
    index=False
)

print("\nMetadata created successfully.")
print(station_metadata.head())

[1/17] Processing Avidipta Housing Complex-Jan 1, 2022-May 30, 2026.xlsx
[2/17] Processing Ballygunge Campus, C.U-Jan 1, 2022-May 30, 2026.xlsx
[3/17] Processing Bethune College-Jan 1, 2022-May 30, 2026.xlsx
[4/17] Processing Dhapa Lock Pumping Station -Jan 1, 2022-May 30, 2026.xlsx
[5/17] Processing East Calcutta Girls College, Lake Town-Jan 1, 2022-May 30, 2026.xlsx
[6/17] Processing Flora Fountain-Jan 1, 2022-May 30, 2026.xlsx
[7/17] Processing Fortis Hospital-Jan 1, 2022-May 30, 2026.xlsx
[8/17] Processing Lady Brabourne College-Jan 1, 2022-May 30, 2026.xlsx
[9/17] Processing Leather Complex-Jan 1, 2022-May 30, 2026.xlsx
[10/17] Processing Lorreto College-Jan 1, 2022-May 30, 2026.xlsx
[11/17] Processing Madhyamgram Municipality-Jan 1, 2022-May 30, 2026.xlsx
[12/17] Processing Maulana Azad Collage-Jan 1, 2022-May 30, 2026.xlsx
[13/17] Processing Presidency University-Jan 1, 2022-May 30, 2026.xlsx
[14/17] Processing Sarojini Naidu College for Women-Jan 1, 2022-May 30, 2026.xlsx
[15/1

In [10]:
import pandas as pd

catalog = pd.DataFrame([
    {
        "dataset_id": "WBPCB_AQI",
        "dataset_name": "WBPCB Air Quality Monitoring",
        "source": "WBPCB",
        "granularity": "Hourly",
        "schema_version": "v1.0"
    },
    {
        "dataset_id": "OM_WEATHER",
        "dataset_name": "Open-Meteo Historical Weather",
        "source": "Open-Meteo",
        "granularity": "Hourly",
        "schema_version": "v1.0"
    },
    {
        "dataset_id": "OM_AQI",
        "dataset_name": "Open-Meteo Air Quality",
        "source": "Open-Meteo",
        "granularity": "Hourly",
        "schema_version": "v1.0"
    }
])

catalog.to_csv(
    "../../data/metadata/data_catalog.csv",
    index=False
)

print("Dataset catalog created.")
print(catalog)

Dataset catalog created.
   dataset_id                   dataset_name      source granularity  \
0   WBPCB_AQI   WBPCB Air Quality Monitoring       WBPCB      Hourly   
1  OM_WEATHER  Open-Meteo Historical Weather  Open-Meteo      Hourly   
2      OM_AQI         Open-Meteo Air Quality  Open-Meteo      Hourly   

  schema_version  
0           v1.0  
1           v1.0  
2           v1.0  
